### ⚒ Setup

In [ ]:
# Install minimal dependencies with pip
# The repository Dockerfile is recommended for routine use.
!pip install cyipopt "pydmf>=1.2.1" "orb-models>=0.7.0" "sella>=v2.4.2" "ase>=3.28.0" "numpy>=2.4.6" "scipy>=1.17.1" "pandas>=3.0.3" "matplotlib>=3.10.9" "seaborn>=0.13.2" "rmsd>=1.6.5" "pillow>=12.2.0"
# Disable JAX GPU preallocation to avoid memory issues
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"


In [ ]:
# Install tblite from source
# 1. Install build tools and dependencies
!pip install meson ninja toml

# 2. Clone the tblite repository
%cd /content
!git clone --depth 1 https://github.com/tblite/tblite.git /opt/tblite

# 3. Build and install tblite binaries and libraries
%cd /opt/tblite
!meson setup _build --prefix=/usr/local -Dpython=true
!meson compile -C _build
!meson install -C _build
%cd /content

# 4. Install Python bindings
!pip install /opt/tblite/python

# 5. Update the library path so CFFI can find the shared library
import os
os.environ['LD_LIBRARY_PATH'] = '/usr/lib64-nvidia:/usr/local/lib:/usr/local/lib/x86_64-linux-gnu'

In [ ]:
# Install PySCF and GPU4PySCF (Colab compatible)
!pip install gpu4pyscf-cuda12x
!pip install "pyscf>=2.13.0"
!python -m cupyx.tools.install_library --cuda 12.x --library cutensor


In [ ]:
# Get MolScout
!git clone https://github.com/hikuram/MolScout.git
!cp -r MolScout/core/* .

In [ ]:
# Configure matplotlib
import matplotlib
import subprocess
import shutil
import logging

# 1. Suppress warnings
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

# 2. Install Microsoft core fonts, including Arial
install_cmd = """
echo "ttf-mscorefonts-installer msttcorefonts/accepted-mscorefonts-eula select true" | debconf-set-selections && \
apt-get update -qq && \
apt-get install -y -qq ttf-mscorefonts-installer
"""
subprocess.run(install_cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# 3. Remove old Matplotlib font cache
cache_dir = matplotlib.get_cachedir()
shutil.rmtree(cache_dir, ignore_errors=True)

# 4. Rebuild the font cache in a subprocess
# This takes about 10-15 seconds.
subprocess.run(["python", "-c", "import matplotlib.pyplot"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("✅ setup complete: font cache rebuilt. Arial is available for !python scripts.")

In [ ]:
# Helper function for file uploads
def upload_as(new_name: str):
    import os
    from google.colab import files
    print(f"Please upload the following file: {new_name}")
    uploaded = files.upload()
    if uploaded:
        uploaded_name = list(uploaded.keys())[0]
        os.rename(uploaded_name, new_name)
        print(f"-> Saved as {new_name}\n")

In [ ]:
# Upload input structures
upload_as("react.xyz")
upload_as("prod.xyz")

### ▶ Run

In [ ]:
# Run the full workflow
!python molscout.py -r react.xyz -p prod.xyz -d result -c 0 -m orbmol

In [ ]:
# Download results
!zip -r /content/download.zip /content/result
from google.colab import files
files.download("/content/download.zip")

### ⚙ Experimental

### 🍣 Running sample_input cases

In [ ]:
!python molscout.py \
-r /content/MolScout/core/sample_input/HCN_HNC/reactant.xyz \
-p /content/MolScout/core/sample_input/HCN_HNC/product.xyz \
-d result/sample_01 -c 0 -m orbmol+alpb

In [ ]:
!python molscout.py \
-r /content/MolScout/core/sample_input/Cl-_CH3F/reactant.xyz \
-p /content/MolScout/core/sample_input/Cl-_CH3F/product.xyz \
-d result/sample_02 -c -1 -m orbmol

In [ ]:
!python molscout.py \
-r /content/MolScout/core/sample_input/NH3_ud/reactant.xyz \
-p /content/MolScout/core/sample_input/NH3_ud/product.xyz \
-d result/sample_03 -c 0 -m orbmol

In [ ]:
!python molscout.py \
-r /content/MolScout/core/sample_input/formic_acid_dimer/reactant.xyz \
-p /content/MolScout/core/sample_input/formic_acid_dimer/product.xyz \
-d result/sample_04 -c 0 -m orbmol

In [ ]:
!python molscout.py \
-r /content/MolScout/core/sample_input/bi_anthracene/reactant.xyz \
-p /content/MolScout/core/sample_input/bi_anthracene/product.xyz \
-d result/sample_05 -c 0 -m orbmol